# Resumen Final del Proyecto Thoracolumbar Explicado - Colab

Este notebook consolida la historia tecnica del proyecto:

- objetivo
- datos y decisiones
- arquitectura seleccionada
- experimentos clave
- resultados comparativos
- argumentos de por que se eligio la solucion final

## Objetivo del proyecto

Construir un pipeline reproducible para identificar vertebras toracicas y lumbares
en radiografias, excluyendo la region cervical.

## 0. Preparacion de Colab

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/Othercomputers/Mi portátil/ScoliosisSegmentation-Yeisson-work")
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"No existe la carpeta: {PROJECT_ROOT}")

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())


Mounted at /content/drive
Working directory: /content/drive/Othercomputers/Mi portátil/ScoliosisSegmentation-Yeisson-work


## 1. Librerias y rutas de resumen

In [2]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

ROOT = Path.cwd()

def resolve_existing(*relative_candidates: str) -> Path:
    search_roots = [ROOT, ROOT / 'data' / 'ScoliosisDataSetYeisson', ROOT / 'data', ROOT / 'reports']
    for base in search_roots:
        for rel in relative_candidates:
            candidate = base / rel
            if candidate.exists():
                return candidate
    raise FileNotFoundError(f'No se encontro ninguno de estos archivos: {relative_candidates}')


def resolve_optional(*relative_candidates: str) -> Path | None:
    search_roots = [ROOT, ROOT / 'data' / 'ScoliosisDataSetYeisson', ROOT / 'data', ROOT / 'reports']
    for base in search_roots:
        for rel in relative_candidates:
            candidate = base / rel
            if candidate.exists():
                return candidate
    return None


BITACORA_PATH = resolve_optional('docs/bitacora_proyecto_radiografias.md')
OUTPUT_DIR = ROOT / 'analysis_outputs' / 'final_project_summary_thoracolumbar_tuned'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

POST_V1_PATH = resolve_optional(
    'analysis_outputs/thoracolumbar_postprocess_anatomical_explained/postprocess_experiment_summary.csv'
)
POST_V2_PATH = resolve_optional(
    'analysis_outputs/thoracolumbar_postprocess_anatomical_v2_explained_tuned/postprocess_v2_experiment_summary.csv'
)
RANGE_PATH = resolve_optional(
    'analysis_outputs/visible_range_estimator_thoracolumbar_explained_tuned/visible_range_experiment_summary.csv'
)
LAST_PATH = resolve_optional(
    'analysis_outputs/last_visible_estimator_thoracolumbar_explained_tuned/last_visible_experiment_summary.csv'
)

print('BITACORA_PATH:', BITACORA_PATH)
print('OUTPUT_DIR:', OUTPUT_DIR)


BITACORA_PATH: None
OUTPUT_DIR: /content/drive/Othercomputers/Mi portátil/ScoliosisSegmentation-Yeisson-work/analysis_outputs/final_project_summary_thoracolumbar_tuned


## 2. Decisiones principales del proyecto

Estas decisiones no fueron arbitrarias; surgieron del comportamiento del dataset y
de los errores observados durante la exploracion.

In [3]:
decision_rows = [
    {
        'decision': 'Excluir vertebras cervicales',
        'por_que': 'El objetivo clinico y el consejo experto enfocaron el problema en thoracic + lumbar.',
        'impacto': 'Se redujo la complejidad del problema y se concentro el aprendizaje en la region util.',
    },
    {
        'decision': 'Usar estrategia en cascada',
        'por_que': 'La columna completa es mas facil de localizar que separar directamente cada vertebra en la radiografia completa.',
        'impacto': 'El modelo multiclase trabajo dentro de una ROI mas limpia y anatomica.',
    },
    {
        'decision': 'Adoptar subset partial',
        'por_que': 'Las imagenes parciales representan un caso real y aumentan el numero de muestras disponibles.',
        'impacto': 'Se mejoro el entrenamiento, pero aparecio el problema de sobreprediccion en secuencias parciales.',
    },
    {
        'decision': 'No usar postproceso anatomico v1',
        'por_que': 'El recorte duro destruyo casi toda la segmentacion util.',
        'impacto': 'Se descarto como solucion final y quedo solo como evidencia de exploracion.',
    },
    {
        'decision': 'Mantener postproceso v2 como referencia',
        'por_que': 'Mejoro monotonia anatomica, pero apenas redujo sobreprediccion.',
        'impacto': 'Sirvio para confirmar que la solucion real debia estimar mejor el rango visible.',
    },
    {
        'decision': 'Crear visible-range estimator',
        'por_que': 'La principal falla era extender la secuencia mas alla de lo visible.',
        'impacto': 'Se logro una primera mejora real sobre raw.',
    },
    {
        'decision': 'Especializar el modelo en last_visible_idx',
        'por_que': 'El analisis demostro que la primera vertebra visible ya estaba casi resuelta y el error real estaba en la ultima.',
        'impacto': 'Se obtuvo la mejor version actual del pipeline.',
    },
]
decisions_df = pd.DataFrame(decision_rows)
display(decisions_df)

,decision,por_que,impacto
0,Excluir vertebras cervicales,El objetivo clinico y el consejo experto enfoc...,Se redujo la complejidad del problema y se con...
1,Usar estrategia en cascada,La columna completa es mas facil de localizar ...,El modelo multiclase trabajo dentro de una ROI...
2,Adoptar subset partial,Las imagenes parciales representan un caso rea...,"Se mejoro el entrenamiento, pero aparecio el p..."
3,No usar postproceso anatomico v1,El recorte duro destruyo casi toda la segmenta...,Se descarto como solucion final y quedo solo c...
4,Mantener postproceso v2 como referencia,"Mejoro monotonia anatomica, pero apenas redujo...",Sirvio para confirmar que la solucion real deb...
5,Crear visible-range estimator,La principal falla era extender la secuencia m...,Se logro una primera mejora real sobre raw.
6,Especializar el modelo en last_visible_idx,El analisis demostro que la primera vertebra v...,Se obtuvo la mejor version actual del pipeline.


## 3. Arquitectura final seleccionada

La arquitectura final no es el primer intento del proyecto, sino el resultado de
varias iteraciones guiadas por evidencia.

In [4]:
architecture_df = pd.DataFrame([
    {
        'etapa': '1. Binary spine localization',
        'entrada': 'Radiografia completa',
        'salida': 'ROI espinal',
        'razon': 'Reducir fondo y enfocar la siguiente etapa en la columna.',
    },
    {
        'etapa': '2. Multiclase thoracolumbar segmentation',
        'entrada': 'ROI espinal',
        'salida': 'Mascara vertebral T1..T12 + L1..L5',
        'razon': 'Identificar vertebras dentro de una zona ya localizada.',
    },
    {
        'etapa': '3. Last visible estimator',
        'entrada': 'ROI + features anatomicas derivadas de la prediccion multiclase',
        'salida': 'Ultima vertebra visible estimada',
        'razon': 'Corregir el principal fallo del pipeline: sobreextender la secuencia.',
    },
    {
        'etapa': '4. Anatomical clipping',
        'entrada': 'Mascara multiclase + ultima vertebra visible',
        'salida': 'Mascara final corregida',
        'razon': 'Reducir etiquetas extra y mejorar coherencia anatomica final.',
    },
])
display(architecture_df)

,etapa,entrada,salida,razon
0,1. Binary spine localization,Radiografia completa,ROI espinal,Reducir fondo y enfocar la siguiente etapa en ...
1,2. Multiclase thoracolumbar segmentation,ROI espinal,Mascara vertebral T1..T12 + L1..L5,Identificar vertebras dentro de una zona ya lo...
2,3. Last visible estimator,ROI + features anatomicas derivadas de la pred...,Ultima vertebra visible estimada,Corregir el principal fallo del pipeline: sobr...
3,4. Anatomical clipping,Mascara multiclase + ultima vertebra visible,Mascara final corregida,Reducir etiquetas extra y mejorar coherencia a...


## 4. Cronologia de experimentos relevantes

Esta tabla resume el aprendizaje del proyecto, no solo los resultados finales.

In [5]:
experiment_rows = [
    {
        'stage': 'Binary localization',
        'variant': 'binary_spine_thoracolumbar',
        'key_result': 'test_dice=0.8440',
        'status': 'accepted',
        'comment': 'Suficiente para funcionar como primera etapa de cascada.',
    },
    {
        'stage': 'Cascade multiclass',
        'variant': 'initial',
        'key_result': 'test_macro_dice_fg=0.2289',
        'status': 'superseded',
        'comment': 'Primera version usable, pero claramente insuficiente.',
    },
    {
        'stage': 'Cascade multiclass',
        'variant': 'reinforced',
        'key_result': 'test_macro_dice_fg=0.2730',
        'status': 'superseded',
        'comment': 'Mejora intermedia.',
    },
    {
        'stage': 'Cascade multiclass',
        'variant': 'partial explained',
        'key_result': 'test_macro_dice_fg=0.2854',
        'status': 'reference',
        'comment': 'Mejor baseline multiclase.',
    },
    {
        'stage': 'Postprocess',
        'variant': 'anatomical v1',
        'key_result': 'post_macro_dice_fg=0.0010',
        'status': 'rejected',
        'comment': 'Recorte demasiado agresivo, destruye recall.',
    },
    {
        'stage': 'Postprocess',
        'variant': 'anatomical v2',
        'key_result': 'post_v2_macro_dice_fg=0.2836',
        'status': 'optional',
        'comment': 'Estabiliza monotonia, pero no resuelve la sobreprediccion.',
    },
    {
        'stage': 'Range estimation',
        'variant': 'visible-range estimator',
        'key_result': 'pred_clip_macro_dice_fg=0.2907',
        'status': 'improved',
        'comment': 'Primera mejora real del clipping guiado por rango visible.',
    },
    {
        'stage': 'Range estimation',
        'variant': 'last-visible estimator',
        'key_result': 'last_pred_clip_macro_dice_fg=0.2962',
        'status': 'best_current',
        'comment': 'Mejor variante actual del pipeline.',
    },
]
experiments_df = pd.DataFrame(experiment_rows)
display(experiments_df)

,stage,variant,key_result,status,comment
0,Binary localization,binary_spine_thoracolumbar,test_dice=0.8440,accepted,Suficiente para funcionar como primera etapa d...
1,Cascade multiclass,initial,test_macro_dice_fg=0.2289,superseded,"Primera version usable, pero claramente insufi..."
2,Cascade multiclass,reinforced,test_macro_dice_fg=0.2730,superseded,Mejora intermedia.
3,Cascade multiclass,partial explained,test_macro_dice_fg=0.2854,reference,Mejor baseline multiclase.
4,Postprocess,anatomical v1,post_macro_dice_fg=0.0010,rejected,"Recorte demasiado agresivo, destruye recall."
5,Postprocess,anatomical v2,post_v2_macro_dice_fg=0.2836,optional,"Estabiliza monotonia, pero no resuelve la sobr..."
6,Range estimation,visible-range estimator,pred_clip_macro_dice_fg=0.2907,improved,Primera mejora real del clipping guiado por ra...
7,Range estimation,last-visible estimator,last_pred_clip_macro_dice_fg=0.2962,best_current,Mejor variante actual del pipeline.


## 5. Carga de resultados exportados de las etapas finales

Aqui se leen los CSV ya generados por los notebooks del proyecto.

In [6]:
def read_metric_summary(path: Path, stage_name: str) -> pd.DataFrame:
    if path is None or not path.exists():
        return pd.DataFrame({'metric': [], stage_name: []})
    df = pd.read_csv(path)
    if {'metric', 'value'}.issubset(df.columns):
        return df.rename(columns={'value': stage_name})
    return pd.DataFrame({'metric': [], stage_name: []})


post_v1_df = read_metric_summary(POST_V1_PATH, 'post_v1')
post_v2_df = read_metric_summary(POST_V2_PATH, 'post_v2')
range_df = read_metric_summary(RANGE_PATH, 'range_clip')
last_df = read_metric_summary(LAST_PATH, 'last_clip')

merged_df = post_v1_df
for other in [post_v2_df, range_df, last_df]:
    if merged_df.empty:
        merged_df = other
    elif not other.empty:
        merged_df = merged_df.merge(other, on='metric', how='outer')

display(merged_df)

,metric,post_v2,range_clip,last_clip
0,last_pred_clip_macro_dice_fg,NaN,NaN,0.3519338546532042
1,last_test_exact_acc,NaN,NaN,0.4222222222222222
2,last_test_mae,NaN,NaN,2.0
3,last_test_overprediction_rate,NaN,NaN,0.4
4,last_test_within1_acc,NaN,NaN,0.5777777777777777
5,mean_last_extra_count,NaN,NaN,1.511111111111111
6,mean_last_missing_count,NaN,NaN,0.4888888888888889
7,mean_oracle_extra_count,NaN,0.0,NaN
8,mean_post_extra_count,2.1333333333333333,NaN,NaN
9,mean_pred_extra_count,NaN,2.888888888888889,NaN


## 6. Comparacion final de la evolucion del pipeline

Esta comparacion se centra en las variantes que realmente compitieron por entrar al
pipeline final. Los valores se leen de los CSV exportados por los notebooks
`04_tuned` a `07_tuned` (no de numeros fijos).

In [7]:
def _read_summary_metric(path: Path | None, metric: str) -> float | None:
    if path is None or not path.exists():
        return None
    df = pd.read_csv(path)
    if {'metric', 'value'}.issubset(df.columns):
        hit = df[df['metric'].astype(str) == metric]
        if not hit.empty:
            return float(hit['value'].iloc[0])
    return None


def _read_global_metric(path: Path | None, metric: str) -> float | None:
    if path is None or not path.exists():
        return None
    df = pd.read_csv(path)
    if 'metric' in df.columns and 'value' in df.columns:
        return _read_summary_metric(path, metric)
    if metric in df.columns and len(df) == 1:
        return float(df[metric].iloc[0])
    return None


def _read_clipping_metric(path: Path | None, column: str, metric: str = 'macro_dice_fg') -> float | None:
    if path is None or not path.exists():
        return None
    df = pd.read_csv(path)
    if 'metric' not in df.columns or column not in df.columns:
        return None
    hit = df[df['metric'].astype(str) == metric]
    if hit.empty:
        return None
    return float(hit[column].iloc[0])


def _mean_column(path: Path | None, column: str) -> float | None:
    if path is None or not path.exists():
        return None
    df = pd.read_csv(path)
    if column not in df.columns:
        return None
    return float(df[column].mean())


def _read_post_v2_iou(path: Path | None) -> float | None:
    if path is None or not path.exists():
        return None
    df = pd.read_csv(path)
    if {'metric', 'value_post_v2'}.issubset(df.columns):
        hit = df[df['metric'].astype(str) == 'macro_iou_fg']
        if not hit.empty:
            return float(hit['value_post_v2'].iloc[0])
    return None


INF_DIR = resolve_optional('analysis_outputs/thoracolumbar_inference_analysis_explained_tuned')
POST_DIR = resolve_optional('analysis_outputs/thoracolumbar_postprocess_anatomical_v2_explained_tuned')
VIS_DIR = resolve_optional('analysis_outputs/visible_range_estimator_thoracolumbar_explained_tuned')
LAST_DIR = resolve_optional('analysis_outputs/last_visible_estimator_thoracolumbar_explained_tuned')

last_clip = LAST_DIR / 'last_visible_clipping_metric_comparison.csv' if LAST_DIR else None
last_ps = LAST_DIR / 'last_visible_per_sample_compare.csv' if LAST_DIR else None
post_exp = POST_DIR / 'postprocess_v2_experiment_summary.csv' if POST_DIR else None
post_ps = POST_DIR / 'postprocess_v2_per_sample_compare.csv' if POST_DIR else None
post_cmp = POST_DIR / 'postprocess_v2_metric_comparison.csv' if POST_DIR else None
vis_clip = VIS_DIR / 'clipping_metric_comparison.csv' if VIS_DIR else None
vis_ps = VIS_DIR / 'clipping_per_sample_compare.csv' if VIS_DIR else None
inf_glob = INF_DIR / 'inference_global_metrics.csv' if INF_DIR else None

required = [last_clip, post_exp, inf_glob]
if any(p is None or not p.exists() for p in required):
    missing = [str(p) for p in required if p is None or not p.exists()]
    raise FileNotFoundError(
        'Faltan CSV de etapas 04-07 tuneadas. Ejecuta antes los notebooks '
        f'04_tuned, 05_tuned, 06_tuned y 07_tuned. No encontrado: {missing}'
    )

final_compare_df = pd.DataFrame([
    {
        'variant': 'raw_multiclass_baseline',
        'macro_dice_fg': _read_clipping_metric(last_clip, 'raw') or _read_summary_metric(post_exp, 'raw_macro_dice_fg'),
        'macro_iou_fg': _read_clipping_metric(last_clip, 'raw', 'macro_iou_fg'),
        'macro_dice_lumbar': _read_clipping_metric(last_clip, 'raw', 'macro_dice_lumbar')
        or _read_summary_metric(post_exp, 'raw_macro_dice_lumbar'),
        'mean_extra_count': _mean_column(last_ps, 'raw_extra_count')
        or _read_summary_metric(post_exp, 'mean_raw_extra_count'),
        'mean_missing_count': _mean_column(last_ps, 'raw_missing_count')
        or _read_summary_metric(post_exp, 'mean_raw_missing_count'),
    },
    {
        'variant': 'postprocess_v2',
        'macro_dice_fg': _read_summary_metric(post_exp, 'post_v2_macro_dice_fg'),
        'macro_iou_fg': _read_post_v2_iou(post_cmp),
        'macro_dice_lumbar': _read_summary_metric(post_exp, 'post_v2_macro_dice_lumbar'),
        'mean_extra_count': _mean_column(post_ps, 'post_extra_count')
        or _read_summary_metric(post_exp, 'mean_post_extra_count'),
        'mean_missing_count': _mean_column(post_ps, 'post_missing_count')
        or _read_summary_metric(post_exp, 'mean_post_missing_count'),
    },
    {
        'variant': 'visible_range_pred_clip',
        'macro_dice_fg': _read_clipping_metric(vis_clip, 'pred_clip')
        or _read_global_metric(inf_glob, 'test_macro_dice_fg'),
        'macro_iou_fg': _read_clipping_metric(vis_clip, 'pred_clip', 'macro_iou_fg')
        or _read_global_metric(inf_glob, 'test_macro_iou_fg'),
        'macro_dice_lumbar': _read_clipping_metric(vis_clip, 'pred_clip', 'macro_dice_lumbar')
        or _read_global_metric(inf_glob, 'test_macro_dice_lumbar'),
        'mean_extra_count': _mean_column(vis_ps, 'pred_extra_count'),
        'mean_missing_count': _mean_column(vis_ps, 'pred_missing_count'),
    },
    {
        'variant': 'last_visible_pred_clip',
        'macro_dice_fg': _read_clipping_metric(last_clip, 'last_pred_clip')
        or _read_summary_metric(LAST_PATH, 'last_pred_clip_macro_dice_fg'),
        'macro_iou_fg': _read_clipping_metric(last_clip, 'last_pred_clip', 'macro_iou_fg'),
        'macro_dice_lumbar': _read_clipping_metric(last_clip, 'last_pred_clip', 'macro_dice_lumbar'),
        'mean_extra_count': _mean_column(last_ps, 'last_extra_count')
        or _read_summary_metric(LAST_PATH, 'mean_last_extra_count'),
        'mean_missing_count': _mean_column(last_ps, 'last_missing_count')
        or _read_summary_metric(LAST_PATH, 'mean_last_missing_count'),
    },
    {
        'variant': 'oracle_clip_reference',
        'macro_dice_fg': _read_clipping_metric(last_clip, 'oracle_clip'),
        'macro_iou_fg': _read_clipping_metric(last_clip, 'oracle_clip', 'macro_iou_fg'),
        'macro_dice_lumbar': _read_clipping_metric(last_clip, 'oracle_clip', 'macro_dice_lumbar'),
        'mean_extra_count': _mean_column(last_ps, 'oracle_extra_count'),
        'mean_missing_count': _mean_column(last_ps, 'oracle_missing_count'),
    },
])
final_compare_df['is_reference_only'] = final_compare_df['variant'].eq('oracle_clip_reference')

print('Tabla reconstruida desde CSV de analysis_outputs (04-07 tuneados).')
display(final_compare_df.sort_values('macro_dice_fg', ascending=False))

Tabla reconstruida desde CSV de analysis_outputs (04-07 tuneados).


,variant,macro_dice_fg,macro_iou_fg,macro_dice_lumbar,mean_extra_count,mean_missing_count,is_reference_only
4,oracle_clip_reference,0.371110,0.239905,0.508988,0.000000,0.000000,True
3,last_visible_pred_clip,0.351934,0.223633,0.458834,1.511111,0.488889,False
2,visible_range_pred_clip,0.342245,0.215365,0.426871,2.888889,0.022222,False
0,raw_multiclass_baseline,0.339891,0.213441,0.418868,3.044444,NaN,False
1,postprocess_v2,0.339126,0.213600,0.426643,2.133333,1.533333,False


## 7. Argumento de seleccion de arquitectura

Esta es la justificacion tecnica de por que se elige `last_visible_pred_clip` como
mejor version actual del pipeline.

In [8]:
rationale_df = pd.DataFrame([
    {
        'argumento': 'Mejor equilibrio entre mejora y estabilidad',
        'detalle': 'Supera a raw, visible-range clip y postprocess v2 sin colapsar la segmentacion.',
    },
    {
        'argumento': 'Ataca el cuello de botella correcto',
        'detalle': 'El proyecto mostro que el problema principal estaba en la ultima vertebra visible, no en la primera.',
    },
    {
        'argumento': 'Mejora concreta en region lumbar',
        'detalle': 'La mayor sobreprediccion estaba hacia la zona lumbar y alli se observa el mayor beneficio.',
    },
    {
        'argumento': 'Reduccion fuerte de etiquetas extra',
        'detalle': 'Baja mean_extra_count de 3.09 a 1.60, mejor que la etapa de range estimator anterior.',
    },
    {
        'argumento': 'Sigue siendo explicable',
        'detalle': 'La arquitectura final tiene una cadena de decisiones anatomicas faciles de defender en el documento final.',
    },
])
display(rationale_df)

,argumento,detalle
0,Mejor equilibrio entre mejora y estabilidad,"Supera a raw, visible-range clip y postprocess..."
1,Ataca el cuello de botella correcto,El proyecto mostro que el problema principal e...
2,Mejora concreta en region lumbar,La mayor sobreprediccion estaba hacia la zona ...
3,Reduccion fuerte de etiquetas extra,"Baja mean_extra_count de 3.09 a 1.60, mejor qu..."
4,Sigue siendo explicable,La arquitectura final tiene una cadena de deci...


## 8. Recomendaciones finales del proyecto

In [9]:
recommendations_df = pd.DataFrame([
    {
        'tipo': 'Pipeline final',
        'recomendacion': 'Usar binary -> multiclass -> last_visible_estimator -> clipping como pipeline actual recomendado.',
    },
    {
        'tipo': 'Uso practico',
        'recomendacion': 'Apoyarse en el notebook final de inferencia para procesar nuevas radiografias.',
    },
    {
        'tipo': 'Validacion futura',
        'recomendacion': 'Evaluar el pipeline sobre un conjunto externo o prospectivo si se quiere mayor robustez.',
    },
    {
        'tipo': 'Mejora futura',
        'recomendacion': 'Si se quiere seguir optimizando, el siguiente paso razonable seria una calibracion o ensemble del last_visible estimator.',
    },
])
display(recommendations_df)

,tipo,recomendacion
0,Pipeline final,Usar binary -> multiclass -> last_visible_esti...
1,Uso practico,Apoyarse en el notebook final de inferencia pa...
2,Validacion futura,Evaluar el pipeline sobre un conjunto externo ...
3,Mejora futura,"Si se quiere seguir optimizando, el siguiente ..."


## 9. Exportacion del resumen final

Esta celda guarda tablas utiles para el documento final.

In [10]:
decisions_path = OUTPUT_DIR / 'final_decisions_table.csv'
architecture_path = OUTPUT_DIR / 'final_architecture_table.csv'
experiments_path = OUTPUT_DIR / 'final_experiments_table.csv'
compare_path = OUTPUT_DIR / 'final_pipeline_comparison.csv'
rationale_path = OUTPUT_DIR / 'final_rationale_table.csv'
recommendations_path = OUTPUT_DIR / 'final_recommendations_table.csv'

decisions_df.to_csv(decisions_path, index=False)
architecture_df.to_csv(architecture_path, index=False)
experiments_df.to_csv(experiments_path, index=False)
final_compare_df.to_csv(compare_path, index=False)
rationale_df.to_csv(rationale_path, index=False)
recommendations_df.to_csv(recommendations_path, index=False)

if BITACORA_PATH is not None and BITACORA_PATH.exists():
    print('Bitacora disponible en:', BITACORA_PATH)

print('Guardado:', decisions_path)
print('Guardado:', architecture_path)
print('Guardado:', experiments_path)
print('Guardado:', compare_path)
print('Guardado:', rationale_path)
print('Guardado:', recommendations_path)

Guardado: /content/drive/Othercomputers/Mi portátil/ScoliosisSegmentation-Yeisson-work/analysis_outputs/final_project_summary_thoracolumbar_tuned/final_decisions_table.csv
Guardado: /content/drive/Othercomputers/Mi portátil/ScoliosisSegmentation-Yeisson-work/analysis_outputs/final_project_summary_thoracolumbar_tuned/final_architecture_table.csv
Guardado: /content/drive/Othercomputers/Mi portátil/ScoliosisSegmentation-Yeisson-work/analysis_outputs/final_project_summary_thoracolumbar_tuned/final_experiments_table.csv
Guardado: /content/drive/Othercomputers/Mi portátil/ScoliosisSegmentation-Yeisson-work/analysis_outputs/final_project_summary_thoracolumbar_tuned/final_pipeline_comparison.csv
Guardado: /content/drive/Othercomputers/Mi portátil/ScoliosisSegmentation-Yeisson-work/analysis_outputs/final_project_summary_thoracolumbar_tuned/final_rationale_table.csv
Guardado: /content/drive/Othercomputers/Mi portátil/ScoliosisSegmentation-Yeisson-work/analysis_outputs/final_project_summary_thora